
# Module 3 — Ungraded Lab (LO3)
**Build a One-Page ROI Model and Impact Dashboard**  
> **Solution Notebook** — worked examples for each PRACTICE CHALLENGE.


In [ ]:

# ================================================
# Setup & Synthetic Inputs
# ================================================
import numpy as np, pandas as pd

rng = np.random.default_rng(11)

# Generic experiment outcome
baseline_rate = 0.12           # e.g., baseline conversion or baseline fraud-loss rate
treatment_lift = 0.03          # +3% relative lift on the good outcome (or -3% on loss, adapt below)
population = 200_000           # eligible events/users per period
adoption = 0.6                 # fraction exposed if launched
gross_margin_per_event = 8.0   # $ margin when the "good" event happens (or avoided loss)
incentive_cost_per_event = 0.80
infra_cost_per_period = 5_000
labeling_cost_per_period = 2_000
retraining_cost_per_period = 1_500
upfront_investment = 25_000    # one-off

# We simulate a "good" outcome (e.g., conversion) lifted upward by 3% relative
# For a loss-avoidance case (fraud), treat the "good" outcome as avoided loss rate times average loss size.
avg_loss_avoided_if_triggered = 18.0  # optional alternative for loss-avoidance framing

period = "monthly"
discount_rate_annual = 0.12
discount_rate_monthly = (1 + discount_rate_annual) ** (1/12) - 1

# Build a small synthetic weekly series for the dashboard (8 weeks)
weeks = pd.date_range("2025-01-06", periods=8, freq="W-MON")
noise = rng.normal(0, 0.005, len(weeks))
weekly_primary_metric = (treatment_lift + noise).clip(-0.01, 0.06)  # proxy for weekly lift drift
weekly_nps_delta = rng.normal(0.01, 0.02, len(weeks))
weekly_latency_ms = rng.normal(420, 40, len(weeks)).clip(250, 650)
weekly_support_tickets = rng.poisson(lam=12, size=len(weeks))

inputs = {
    "baseline_rate": baseline_rate,
    "treatment_lift": treatment_lift,
    "population": population,
    "adoption": adoption,
    "gross_margin_per_event": gross_margin_per_event,
    "incentive_cost_per_event": incentive_cost_per_event,
    "infra_cost_per_period": infra_cost_per_period,
    "labeling_cost_per_period": labeling_cost_per_period,
    "retraining_cost_per_period": retraining_cost_per_period,
    "upfront_investment": upfront_investment,
    "discount_rate_monthly": float(discount_rate_monthly)
}

pd.DataFrame({"weeks": weeks, "primary_lift": weekly_primary_metric,
              "nps_delta": weekly_nps_delta,
              "latency_ms": weekly_latency_ms,
              "support_tickets": weekly_support_tickets}).head()


In [ ]:

# ================================================
# Helper Functions
# ================================================
import numpy as np, pandas as pd
from math import ceil
from typing import Dict, List
from numpy.random import default_rng

def lift_to_revenue(baseline_rate, lift, population, adoption, margin_per_event):
    """Translate lift into incremental revenue per period."""
    # Assumes lift is relative on the good outcome; if it's loss-avoidance, interpret accordingly.
    treated_events = population * adoption
    baseline_events = baseline_rate * treated_events
    treatment_events = baseline_events * (1 + lift)
    delta_events = treatment_events - baseline_events
    delta_revenue = delta_events * margin_per_event
    return float(delta_revenue), float(delta_events)

def total_costs(incentive_per_event, delta_events, infra, labeling, retraining):
    variable = incentive_per_event * max(delta_events, 0.0)
    total = variable + infra + labeling + retraining
    return float(total), float(variable)

def payback_period(monthly_net_cashflow: float, upfront: float, max_months:int=36):
    """Return months to payback (ceil), or None if not reached in horizon."""
    if monthly_net_cashflow <= 0:
        return None
    months = ceil(upfront / monthly_net_cashflow)
    return months if months <= max_months else None

def npv_of_stream(monthly_net_cashflow: float, months:int, monthly_discount: float, upfront: float=0.0):
    """NPV of level monthly net cashflows over a horizon, minus upfront."""
    if months <= 0:
        return -float(upfront)
    disc = [(monthly_net_cashflow / ((1 + monthly_discount) ** t)) for t in range(1, months+1)]
    return float(sum(disc) - upfront)

def sensitivity_scan(base_inputs: Dict, ranges: Dict[str, float], months:int=12):
    """Vary each input by ±pct in 'ranges' and compute NPV; return ranked impact."""
    out = []
    for k, pct in ranges.items():
        for direction in (-1, +1):
            trial = dict(base_inputs)
            if k in ("infra_cost_per_period","labeling_cost_per_period","retraining_cost_per_period",
                     "gross_margin_per_event","incentive_cost_per_event","baseline_rate","treatment_lift","adoption"):
                trial[k] = base_inputs[k] * (1 + direction * pct)
            # recompute
            delta_rev, delta_events = lift_to_revenue(
                trial["baseline_rate"], trial["treatment_lift"],
                trial["population"], trial["adoption"], trial["gross_margin_per_event"]
            )
            tot_cost, var_cost = total_costs(trial["incentive_cost_per_event"], delta_events,
                                             trial["infra_cost_per_period"], trial["labeling_cost_per_period"],
                                             trial["retraining_cost_per_period"])
            incr_profit = delta_rev - tot_cost
            npv = npv_of_stream(incr_profit, months, trial["discount_rate_monthly"], trial["upfront_investment"])
            out.append({"driver": k, "direction": "minus" if direction==-1 else "plus", "NPV": float(npv)})
    df = pd.DataFrame(out)
    # compute swing for each driver
    swings = df.pivot(index="driver", columns="direction", values="NPV")
    swings["swing"] = (swings["plus"] - swings["minus"]).abs()
    swings = swings.sort_values("swing", ascending=False)
    return swings.reset_index(), df

def bootstrap_ci_for_profit(baseline_rate, lift, population, adoption, margin_per_event,
                            incentive_per_event, infra, labeling, retraining,
                            n_boot=500, seed=7):
    """Bootstrap CI for incremental profit by resampling Bernoulli outcomes and ARPU-like noise."""
    rng = default_rng(seed)
    treated = int(population * adoption)
    base_events = rng.binomial(treated, baseline_rate, size=n_boot)
    treat_events = rng.binomial(treated, min(1.0, baseline_rate*(1+lift)), size=n_boot)
    delta_events = (treat_events - base_events).astype(float)
    # Margin noise (optional)
    margin = rng.normal(margin_per_event, 0.8, size=n_boot).clip(0, None)
    delta_revenue = delta_events * margin
    var_cost = np.maximum(delta_events, 0.0) * incentive_per_event
    incr_profit = delta_revenue - (var_cost + infra + labeling + retraining)
    lo, hi = np.percentile(incr_profit, [2.5, 97.5])
    return float(incr_profit.mean()), float(lo), float(hi)
print("Helpers ready.")



## Activity 1 — Inputs & Lift → Dollars
Compute incremental revenue/savings from lift.


In [ ]:

### SOLUTION — Activity 1
delta_revenue, delta_events = lift_to_revenue(
    inputs["baseline_rate"], inputs["treatment_lift"],
    inputs["population"], inputs["adoption"], inputs["gross_margin_per_event"]
)
pd.DataFrame([{
    "baseline_rate": inputs["baseline_rate"],
    "treatment_lift": inputs["treatment_lift"],
    "treated_population": inputs["population"] * inputs["adoption"],
    "delta_events": delta_events,
    "delta_revenue": delta_revenue
}])



## Activity 2 — One-Page ROI (Incremental Profit, Payback, NPV)
Build the compact table executives want to see.


In [ ]:

### SOLUTION — Activity 2
total_cost, var_cost = total_costs(
    inputs["incentive_cost_per_event"], delta_events,
    inputs["infra_cost_per_period"], inputs["labeling_cost_per_period"], inputs["retraining_cost_per_period"]
)
incremental_profit = delta_revenue - total_cost
payback_months = payback_period(incremental_profit, inputs["upfront_investment"], max_months=36)
npv_12m = npv_of_stream(incremental_profit, 12, inputs["discount_rate_monthly"], inputs["upfront_investment"])

pd.DataFrame([{
    "delta_revenue": delta_revenue,
    "variable_cost": var_cost,
    "fixed_costs": total_cost - var_cost,
    "incremental_profit": incremental_profit,
    "payback_months": payback_months,
    "npv_12m": npv_12m
}])



## Activity 3 — Sensitivity Analysis
Rank drivers by their impact on NPV.


In [ ]:

### SOLUTION — Activity 3
ranges = {"treatment_lift":0.20,"gross_margin_per_event":0.20,"incentive_cost_per_event":0.20,"infra_cost_per_period":0.20,"adoption":0.20}
swings, raw = sensitivity_scan(inputs, ranges, months=12)
swings


In [ ]:

import matplotlib.pyplot as plt
plt.figure()
plt.barh(swings["driver"], swings["swing"])
plt.xlabel("NPV swing (absolute)"); plt.ylabel("Driver"); plt.title("Sensitivity Tornado — NPV Swing by Driver (± range)")
plt.show()



## Activity 4 — Impact Dashboard (Primary + Guardrails)
Create minimal weekly trends and a go/no-go.


In [ ]:

### SOLUTION — Activity 4
import matplotlib.pyplot as plt
weekly = pd.date_range("2025-01-06", periods=8, freq="W-MON")
primary = pd.Series([v for v in np.linspace(inputs["treatment_lift"]*0.8, inputs["treatment_lift"]*1.1, 8)], index=weekly)
plt.figure(); plt.plot(primary.index, primary.values); plt.xlabel("Week"); plt.ylabel("Primary metric (lift proxy)"); plt.title("Weekly Primary Metric Trend"); plt.show()

nps = pd.Series(np.linspace(0.0, 0.02, 8), index=primary.index)
latency = pd.Series(np.linspace(470, 510, 8), index=primary.index)
tickets = pd.Series(np.linspace(10, 14, 8), index=primary.index)

plt.figure(); plt.plot(nps.index, nps.values); plt.xlabel("Week"); plt.ylabel("NPS delta"); plt.title("NPS Delta (weekly)"); plt.show()
plt.figure(); plt.plot(latency.index, latency.values); plt.xlabel("Week"); plt.ylabel("Latency (ms)"); plt.title("Approval Latency (weekly)"); plt.show()
plt.figure(); plt.plot(tickets.index, tickets.values); plt.xlabel("Week"); plt.ylabel("Support tickets"); plt.title("Support Tickets (weekly)"); plt.show()



## Activity 5 — Decision Rules
Make thresholds explicit; output a GO/NO-GO.


In [ ]:

### SOLUTION — Activity 5
thresholds = {"roi_min":0.0,"payback_max_months":6,"nps_min_delta":0.0,"latency_max_ms":500,"tickets_max_increase":2}
roi_ok = (incremental_profit >= 0)
payback_ok = (payback_months is not None) and (payback_months <= thresholds["payback_max_months"])
nps_ok = (0.02 - 0.0) >= thresholds["nps_min_delta"]
latency_ok = 510 <= thresholds["latency_max_ms"]
tickets_ok = (14 - 10) <= thresholds["tickets_max_increase"]
decision = "GO" if all([roi_ok, payback_ok, nps_ok, tickets_ok, latency_ok]) else "NO-GO"
pd.DataFrame([{"decision": decision, "ROI >= 0": roi_ok, "Payback <= 6 mo": payback_ok, "NPS Δ >= 0": nps_ok, "Latency <= 500 ms": latency_ok, "Tickets not higher than +2": tickets_ok}])



## Activity 6 — Uncertainty & Post-Launch Monitoring
Quantify uncertainty and outline monitors.


In [ ]:

### SOLUTION — Activity 6
mean_profit, lo, hi = bootstrap_ci_for_profit(
    inputs["baseline_rate"], inputs["treatment_lift"], inputs["population"], inputs["adoption"],
    inputs["gross_margin_per_event"], inputs["incentive_cost_per_event"],
    inputs["infra_cost_per_period"], inputs["labeling_cost_per_period"], inputs["retraining_cost_per_period"],
    n_boot=400, seed=13
)
pd.DataFrame([{"mean_incremental_profit": mean_profit, "ci_95_lo": lo, "ci_95_hi": hi}])


> End of Solution.